# Tune Periodicity Thresholds (Canonical)

This notebook is the single canonical periodicity-tuning workflow for MALCA. It supersedes the old `tune_periodicity*.py` variants by combining:

- Periodic injections (sine, narrow eclipse)
- One-off event injections (single dip, single jump)
- LS bootstrap significance, PDM SNR, CE SNR, and morphology fit strength

Outputs are written to `output/diagnostics/` for reproducibility across machines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from malca.core.periodogram import pdm_find_period, ce_find_period
from malca.core.stats import bootstrap_lomb_scargle
from malca.events import classify_run_morphology

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)

## 1) Simulation Helpers

In [ ]:
def generate_cadence(n_points=2000, duration_years=10, seasonal_gap_days=100):
    """Generate a realistic long-baseline cadence with seasonal gaps."""
    total_days = duration_years * 365.25
    jd = np.sort(np.random.uniform(0, total_days, size=int(n_points * 1.5)))
    is_observable = (jd % 365.25) <= (365.25 - seasonal_gap_days)
    jd = jd[is_observable]
    if len(jd) > n_points:
        jd = np.sort(np.random.choice(jd, size=n_points, replace=False))
    return jd + 2456000.0


def inject_sine(jd, base_mag, noise, period, amplitude):
    signal = amplitude * np.sin(2 * np.pi * jd / period)
    return base_mag + signal + noise


def inject_eclipse(jd, base_mag, noise, period, depth, width_phase=0.05):
    phase = (jd % period) / period
    sigma_phase = width_phase / 2.355  # FWHM -> sigma
    signal = depth * np.exp(-0.5 * ((phase - 0.5) / sigma_phase) ** 2)
    return base_mag + signal + noise


def inject_single_dip(jd, base_mag, noise, depth, t0, sigma_days):
    signal = depth * np.exp(-((jd - t0) ** 2) / (2 * sigma_days ** 2))
    return base_mag + signal + noise


def inject_single_jump(jd, base_mag, noise, amp, t0, tE_days):
    u = (jd - t0) / tE_days
    u = np.where(u == 0, 1e-5, u)
    magnification = (u ** 2 + 2) / (np.abs(u) * np.sqrt(u ** 2 + 4)) - 1
    signal = amp * magnification
    return base_mag + signal + noise

## 2) Metric Evaluators

In [ ]:
def evaluate_pdm(jd, mag, min_period=0.1, max_period=100.0, n_periods=2000):
    best_p, periods, thetas = pdm_find_period(
        jd, mag, min_period=min_period, max_period=max_period, n_periods=n_periods
    )
    min_theta = float(np.min(thetas))
    std_theta = float(np.std(thetas))
    pdm_snr = np.nan if std_theta == 0 else float((np.mean(thetas) - min_theta) / std_theta)
    return float(best_p), min_theta, pdm_snr


def evaluate_ce(jd, mag, min_period=0.1, max_period=100.0, n_periods=2000):
    best_p, periods, entropies = ce_find_period(
        jd, mag, min_period=min_period, max_period=max_period, n_periods=n_periods
    )
    min_entropy = float(np.min(entropies))
    std_entropy = float(np.std(entropies))
    ce_snr = np.nan if std_entropy == 0 else float((np.mean(entropies) - min_entropy) / std_entropy)
    return float(best_p), min_entropy, ce_snr


def evaluate_ls(jd, mag, err, n_bootstrap=100):
    res = bootstrap_lomb_scargle(jd, mag, err, n_bootstrap=n_bootstrap)
    if not isinstance(res, dict):
        return np.nan, np.nan
    return float(res.get("ls_period_days", np.nan)), float(res.get("ls_bootstrap_sig", np.nan))


def evaluate_morphology(jd, mag, err, t0, kind):
    center_idx = int(np.argmin(np.abs(jd - t0)))
    start_i = max(0, center_idx - 15)
    end_i = min(len(jd) - 1, center_idx + 15)
    run_idx = np.arange(start_i, end_i + 1)

    baseline = np.full_like(mag, np.nanmedian(mag))
    try:
        res = classify_run_morphology(jd, mag, err, run_idx, baseline=baseline, kind=kind)
    except Exception:
        return np.nan, "error"

    if not isinstance(res, dict):
        return np.nan, "unknown"

    return float(res.get("delta_bic_null", np.nan)), str(res.get("morphology", "unknown"))


def first_crossing_amplitude(df, column, threshold, lower_is_better):
    values = df[["amplitude_mag", column]].dropna()
    if values.empty:
        return np.nan

    if lower_is_better:
        hit = values[values[column] <= threshold]
    else:
        hit = values[values[column] >= threshold]

    if hit.empty:
        return np.nan
    return float(hit["amplitude_mag"].min())

## 3) Run Consolidated Grid Search

In [ ]:
jd = generate_cadence()
baseline_mag = 15.0
noise_sigma = 0.05
err = np.full(len(jd), noise_sigma)
noise = np.random.normal(loc=0.0, scale=noise_sigma, size=len(jd))

inject_period = 3.14159
t0_event = float(np.median(jd))
sigma_event = 5.0
tE_event = 10.0
test_amplitudes = np.unique(
    np.concatenate([
        np.linspace(0.01, 0.30, 12),
        np.linspace(0.35, 5.00, 20),
    ])
)

model_specs = [
    ("periodic_sine", "dip", lambda amp: inject_sine(jd, baseline_mag, noise, inject_period, amp)),
    ("periodic_eclipse", "dip", lambda amp: inject_eclipse(jd, baseline_mag, noise, inject_period, amp)),
    ("single_dip", "dip", lambda amp: inject_single_dip(jd, baseline_mag, noise, amp, t0_event, sigma_event)),
    ("single_jump", "jump", lambda amp: inject_single_jump(jd, baseline_mag, noise, -amp, t0_event, tE_event)),
]

rows = []
print("Running consolidated tuning grid...")
for amp in test_amplitudes:
    for model_name, morph_kind, generator in model_specs:
        mag = generator(amp)

        ls_period, ls_bootstrap_sig = evaluate_ls(jd, mag, err, n_bootstrap=100)
        pdm_period, pdm_min_theta, pdm_snr = evaluate_pdm(jd, mag)
        ce_period, ce_min_entropy, ce_snr = evaluate_ce(jd, mag)
        delta_bic_null, morphology = evaluate_morphology(jd, mag, err, t0_event, kind=morph_kind)

        rows.append({
            "model": model_name,
            "amplitude_mag": float(amp),
            "ls_period_days": ls_period,
            "ls_bootstrap_sig": ls_bootstrap_sig,
            "pdm_period_days": pdm_period,
            "pdm_min_theta": pdm_min_theta,
            "pdm_snr": pdm_snr,
            "ce_period_days": ce_period,
            "ce_min_entropy": ce_min_entropy,
            "ce_snr": ce_snr,
            "morph_delta_bic_null": delta_bic_null,
            "morphology": morphology,
        })

df_results = pd.DataFrame(rows).sort_values(["model", "amplitude_mag"]).reset_index(drop=True)
print(f"Done. Generated {len(df_results)} model/amplitude rows.")

summary_rows = []
for model_name in ["periodic_sine", "periodic_eclipse", "single_dip", "single_jump"]:
    d = df_results[df_results["model"] == model_name].sort_values("amplitude_mag")

    summary_rows.append({
        "model": model_name,
        "amp_at_ls_sig_0p01": first_crossing_amplitude(d, "ls_bootstrap_sig", 0.01, lower_is_better=True),
        "amp_at_pdm_snr_4": first_crossing_amplitude(d, "pdm_snr", 4.0, lower_is_better=False),
        "amp_at_ce_snr_4": first_crossing_amplitude(d, "ce_snr", 4.0, lower_is_better=False),
        "amp_at_morph_delta_bic_10": first_crossing_amplitude(d, "morph_delta_bic_null", 10.0, lower_is_better=False),
        "most_common_morphology": d["morphology"].mode().iat[0] if not d["morphology"].mode().empty else "unknown",
    })

threshold_summary = pd.DataFrame(summary_rows)
threshold_summary

In [ ]:
out_dir = Path("output/diagnostics")
out_dir.mkdir(parents=True, exist_ok=True)

threshold_summary.to_parquet(out_dir / "tune_periodicity_threshold_summary.parquet", index=False)
df_results.to_parquet(out_dir / "tune_periodicity_grid_results.parquet", index=False)

print("Wrote:")
print(out_dir / "tune_periodicity_threshold_summary.parquet")
print(out_dir / "tune_periodicity_grid_results.parquet")

## 4) Diagnostic Plots

In [ ]:
from malca.plotting.lightcurve_publication import finalize_publication_figure
model_order = ["periodic_sine", "periodic_eclipse", "single_dip", "single_jump"]
title_map = {
    "periodic_sine": "Periodic Sine",
    "periodic_eclipse": "Periodic Narrow Eclipse",
    "single_dip": "Single Dip (One-off)",
    "single_jump": "Single Jump (One-off)",
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True)
for ax, model_name in zip(axes.flat, model_order):
    d = df_results[df_results["model"] == model_name].sort_values("amplitude_mag")

    ax.plot(d["amplitude_mag"], d["pdm_snr"], marker="o", label="PDM SNR")
    ax.plot(d["amplitude_mag"], d["ce_snr"], marker="s", label="CE SNR")
    ax.axhline(4.0, color="tab:red", linestyle="--", label="SNR=4")
    ax.set_xlim(0.0, 5.0)
    ax.set_title(title_map[model_name])
    ax.set_ylabel("PDM/CE SNR")
    ax.grid(alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(d["amplitude_mag"], d["ls_bootstrap_sig"], marker="^", color="tab:green", label="LS bootstrap sig")
    ax2.axhline(0.01, color="darkgreen", linestyle=":", label="sig=0.01")
    ax2.set_ylabel("LS bootstrap sig", color="tab:green")
    ax2.tick_params(axis="y", labelcolor="tab:green")
    ax2.set_ylim(-0.02, 1.02)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8)

for ax in axes[-1, :]:
    ax.set_xlabel("Injected amplitude/depth (mag)")

finalize_publication_figure(fig)
metrics_path = out_dir / "tune_periodicity_metrics.png"
fig.savefig(metrics_path, dpi=300)
print(f"Saved {metrics_path}")


In [ ]:
from malca.plotting.lightcurve_publication import (
    figsize_from_legacy,
    finalize_publication_figure,
)
fig, axes = plt.subplots(2, 2, figsize=figsize_from_legacy(16, 10), sharex=True)
for ax, model_name in zip(axes.flat, model_order):
    d = df_results[df_results["model"] == model_name].sort_values("amplitude_mag")

    ax.plot(d["amplitude_mag"], d["morph_delta_bic_null"], marker="D", color="tab:purple")
    ax.axhline(10.0, color="tab:red", linestyle="--", label="Delta BIC = 10")
    ax.set_xlim(0.0, 5.0)
    ax.set_title(title_map[model_name])
    ax.set_ylabel("Morphology delta_bic_null")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")

for ax in axes[-1, :]:
    ax.set_xlabel("Injected amplitude/depth (mag)")

finalize_publication_figure(fig)
morph_path = out_dir / "tune_periodicity_morphology.png"
fig.savefig(morph_path, dpi=300)
print(f"Saved {morph_path}")
